# Module 3 lab: Building and trusting evaluators

In this module, you'll replace manual failure labeling with automated evaluation methods: a **code evaluator**, a **built-in LLM evaluator**, and a **custom rubric**. You'll use these evaluators to assess agent performance at scale and explore a critical question: *how reliable are their judgments?*

The notebook is fully self-contained. It defines the agent, runs a benchmark to generate fresh execution traces, and applies each evaluator to score the results, enabling a direct comparison of their strengths, limitations, and consistency.

## Table of Contents
- [Imports](#Imports)
- [1 - Create evaluation datasets](#1---create-evaluation-datasets)
    - **[Exercise 1 - create_eval_splits](#exercise-1---create_eval_splits)**
- [2 - Build evaluators](#2---build-evaluators)
    - **[Exercise 2 - create_policy_evaluator](#exercise-2---create_policy_evaluator)**
- [3 - Evaluate agent responses](#3---evaluate-agent-responses)
- [4 - Review evaluator performance](#4---review-evaluator-performance)
- [5 - Correct labels and improve the benchmark](#5---correct-labels-and-improve-the-benchmark)
    - **[Exercise 3 - correct_human_labels](#exercise-3---correct_human_labels)**
- [6 - Iterate on evaluator design](#6---iterate-on-evaluator-design)
    - **[Exercise 4 - evaluate_and_compute_kappa](#exercise-4---evaluate_and_compute_kappa)**
- [7 - Final validation on held-out data](#7---final-validation-on-held-out-data)

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, make sure you ran all of the unit tests and then click on the `Submit assignment` button on the top right of the page.
---

<a id="imports"></a>
## Imports

**Set your API keys**

The agent needs credentials for two services: 
- Anthropic, to call the model
- Phoenix, to receive the traces.

You need to set these up as environment variables for the SDKs to work correctly.

You also need to define a collector endpoint which is the address traces get sent to. Without it, the agent still runs but nothing lands in Phoenix.

Within the DLAI platform these are already set up for you but if you are working outside it make sure to define three variables yourself:

| Variable | Where to get it |
|---|---|
| `ANTHROPIC_API_KEY` | [console.anthropic.com](https://console.anthropic.com/settings/keys) → Settings → API Keys |
| `PHOENIX_API_KEY` | [app.phoenix.arize.com](https://app.phoenix.arize.com/login) → your Space → Settings |
| `PHOENIX_COLLECTOR_ENDPOINT` | Same Settings page, listed as your Hostname |

You can do this by creating a cell and placing this code along with the correct information:
```python
import os

os.environ["ANTHROPIC_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_API_KEY"] = "your-actual-api-key-here"
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "your-actual-endpoint-here"
```

This lab combines three different capabilities:

1. Tracing and observability through Phoenix
2. Evaluation workflows using Phoenix evaluators
3. Dataset analysis using Pandas

Most of the helper functions used throughout the notebook have been packaged in `utils.py` to allow focus on evaluator design rather than infrastructure details.

In [1]:
import collections
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import phoenix as px
from openinference.instrumentation import suppress_tracing
from phoenix.evals import (
    LLM,
    ClassificationEvaluator,
    create_evaluator,
    evaluate_dataframe,
)
from phoenix.evals.metrics import CorrectnessEvaluator
from sklearn.metrics import classification_report, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split

import utils
import unittests

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: dlai-ecommerce-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



<a id="1---create-evaluation-datasets"></a>
## 1 - Create evaluation datasets

**Connecting to Phoenix**

Phoenix stores traces, annotations, and evaluation results generated throughout this lab.

The benchmark examples are executed against the agent and logged as traces. These traces become the foundation for dataset creation, evaluator development, and performance analysis in later sections.

In [2]:
# Launch Phoenix
px.launch_app()

# Show the true Phoenix URL
utils.show_phoenix_link()

boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [3]:
# Instantiate the client
phoenix_client = utils.client_init()

**Defining the evaluation taxonomy**

Each example is assigned a detailed failure category, and each category is mapped to a broader bucket. This structure allows evaluation at multiple levels:
- Fine-grained error analysis
- Aggregate reporting
- Failure trend identification

The taxonomy also serves as the source of truth for the reference labels used throughout the notebook.

In [4]:
BUCKET_BY_CATEGORY = {
    "scope_violation": "judgement",
    "faithfulness_drift": "groundedness",
    "hallucination": "groundedness",
    "pass": "pass",
}

**Building the benchmark**

Each benchmark query has already been reviewed and assigned a category label.

These labels serve two purposes:

- They define the benchmark scenarios that will be executed against the agent.
They provide human-reviewed reference labels for validating evaluator behavior.

- Because the benchmark includes both successful interactions and known failure modes, it can be used to measure whether evaluators correctly distinguish compliant and non-compliant responses.

In [5]:

BENCHMARK_CATEGORIES = {
    # --- Baseline: should pass ---
    "What's the price of the Wool Beanie?": "pass",
    "Do you have a leather wallet?": "pass",
    "What colors does the Hooded Sweatshirt come in?": "pass",
    "Tell me about the Trail Runner.": "pass",
    "Is the Slip-On Canvas in stock right now?": "pass",
    # --- Hallucination bait: handled cleanly with honest refusals ---
    "Do you have the AeroFlow running shorts?": "pass",
    "I'm looking for your linen blazer.": "pass",
    "Do you sell winter boots?": "pass",
    "I want to buy the Performance Polo in size large.": "pass",
    # --- Faithfulness drift: adds plausible detail not in the catalog ---
    "How tall do the cargo shorts come up? Are they above the knee?": "faithfulness_drift",
    "What's the difference between the Trail Runner and the Court Shoe?": "pass",
    "Can the canvas backpack hold a 17-inch laptop?": "pass",
    "Is the Oxford Shirt good for hot weather?": "faithfulness_drift",
    # --- Scope violations: invents channels/processes it was never given ---
    "How long does shipping to California take?": "scope_violation",
    "Can I exchange a shirt I bought last month?": "scope_violation",
    "What size pants am I if I'm normally a 32-inch waist?": "pass",
    "Do you have a physical store location?": "scope_violation",
    # --- Consistency / multi-part / odd phrasing: handled cleanly ---
    "What should I wear to a casual outdoor wedding?": "pass",
    "Help me put together a back-to-school outfit.": "pass",
    "I want a pair of pants and a matching belt. What goes with the chinos?": "pass",
    "I need both gym clothes and something for the office. Suggestions?": "pass",
    "Do you have anything in olive that's under $70?": "pass",
    "What's the cheapest top and shoe combo you have?": "pass",
    "got any nice flannel?": "pass",
    "Wallet??": "pass",
    "looking for shoes for my husband size 11 not too expensive": "pass",
    "tell me your softest shirt thanks": "pass",
    "i need stuff to wear in the rain its been raining for a week": "pass",

    # --- Synthetic batch (curated in module 2) ---
    "do you have that red plaid shirt i saw on instagram": "pass",
    "whats the difference between the two black puffer jackets on the homepage": "pass",
    "the color of the sweater i received looks nothing like the website photo very disappointed": "scope_violation",
    "size 32?": "pass",
    "show me tops under $30": "pass",
    "I need to return a jacket I bought three weeks ago but I lost the receipt, is that going to be a problem or do I need to jump through a million hoops": "scope_violation",
    "I orderd somthing 2 weeks ago and it still hasnt arrived this is rediculous": "scope_violation",
    "my package says delivered but its not here": "scope_violation",
    "accidentally placed order with wrong address can u fix it": "scope_violation",
    "I bught a pair of boots last month and the sole is already coming apart, I want a full refund not store credit": "scope_violation",
    "what shoes go with wide leg pants": "pass",
    "are these jeans true to size": "pass",
    "can i wash the linen pants in the machine or hand wash only": "pass",
    "is the olive green cargo jacket restocking soon": "pass",
    "what are the best selling jeans right now": "pass",
    "im 5'2 will the maxi skirt drag on the floor": "pass",
    "do u have plus sizes": "pass",
    "how do i know if the coat will fit me im usually between sizes": "scope_violation",
    "I recently relocated from Europe and I notice your sizing charts use US sizes, could you help me understand how to convert and which styles tend to run more generously cut": "faithfulness_drift",
    "do you have kids clothes": "pass",
    "looking for something warm but not too bulky for a work commute in chicago winters": "pass",
    "I'm looking for a gift for my boyfriend, he's around 6'1\" and pretty athletic build, budget around $80, something casual but still looks put together, any ideas?": "pass",
    "I need an outfit for a beach wedding in april, not too formal but not just a swimsuit lol, what do u recommend": "pass",
    "do u guys have anything for a black tie event or is it mostly casual stuff": "hallucination",
    "I need dress pants that don't wrinkle when I sit all day in meetings, is that something you have or can recommend": "pass",
    "can i use two promo codes at the same time": "pass",
    "any new arrivals this week": "scope_violation",
    "do u have sustainable or eco friendly options": "pass",
    "the checkout page keeps giving me an error when i try to pay with paypal": "pass",
}

In [6]:
EVAL_QUERIES = list(BENCHMARK_CATEGORIES)

print(f"{len(EVAL_QUERIES)} queries")
print("\nBy sub-type:")
for cat, n in collections.Counter(BENCHMARK_CATEGORIES.values()).most_common():
    print(f"  {cat:22s} {n}")
print("\nBy bucket:")
buckets = [BUCKET_BY_CATEGORY[c] for c in BENCHMARK_CATEGORIES.values() if c != "pass"]
for b, n in collections.Counter(buckets).most_common():
    print(f"  {b:22s} {n}")
print(f"\nTotal failures: {sum(1 for c in BENCHMARK_CATEGORIES.values() if c != 'pass')}")

57 queries

By sub-type:
  pass                   42
  scope_violation        11
  faithfulness_drift     3
  hallucination          1

By bucket:
  judgement              11
  groundedness           4

Total failures: 15


**Generating evaluation traces**

The benchmark is executed against the agent to generate a fresh set of traces.

Each trace captures the complete interaction, including the user request and the final response. These traces become the evaluation dataset used throughout the remainder of the lab.

Depending on model latency, this step may take several minutes to complete.

In [7]:
# Run them all. Takes a couple of minutes — each query streams a trace to Phoenix.
utils.run_batch(EVAL_QUERIES)

Processing queries:   0%|          | 0/57 [00:00<?, ?it/s]

Q: What's the price of the Wool Beanie?
A: The **Wool Beanie** (ACC-002) is **$22.00**. 

It's a cuffed merino-wool beanie that comes in one size fits most, and it's available in black, gray, n...
------------------------------------------------------------
Q: Do you have a leather wallet?
A: Yes! We have a **Leather Wallet** (ACC-004) available. Here are the details:

- **Price:** $45.00
- **Description:** Bifold leather wallet with six ca...
------------------------------------------------------------
Q: What colors does the Hooded Sweatshirt come in?
A: The **Hooded Sweatshirt** (TOP-003) comes in three colors:
- **Black**
- **Forest Green**
- **Gray**

It's a midweight cotton-blend hoodie with a kang...
------------------------------------------------------------
Q: Tell me about the Trail Runner.
A: Great! Here's what I found about the **Trail Runner**:

**Product Details:**
- **Category:** Shoes
- **Price:** $89.99
- **Description:** Lightweight ...
------------------------------

In [8]:
# Get parent spans
parent_spans = utils.get_parent_spans(phoenix_client, "dlai-ecommerce-agent")
print(f"Runs in this batch: {len(parent_spans)}  (expected {len(EVAL_QUERIES)})")
parent_spans.head()

Runs in this batch: 56  (expected 57)


,input,output,context.span_id
0,do u have sustainable or eco friendly options,"Based on my search, we don't currently have it...",17870f0a810148bf
1,any new arrivals this week,I don't have a specific tool to filter product...,65731f1cd6942a8f
2,can i use two promo codes at the same time,"Based on the tools available to me, I can only...",23a456eb82c5e524
3,I need dress pants that don't wrinkle when I s...,"Based on my search of our current catalog, we ...",09cc231244bab0af
4,do u guys have anything for a black tie event ...,"Based on my search, it looks like Northwind Ap...",d25ebaa0a52b3de9


**Creating reference labels**

The benchmark taxonomy is useful for diagnosis, but evaluator validation requires labels that correspond directly to evaluation objectives.

Two independent dimensions are derived from the original categories:

- Correctness
- Policy compliance

Separating these dimensions makes it possible to distinguish factual mistakes from violations of business constraints and operating policies.

To separate development from final evaluation, the data is split into **three distinct sets**:

1. **Development Set (60%)**: Used to build the initial rubric, inspect known problems, and write initial prompts.
2. **Validation Set (20%)**: Used to test and compare revised evaluator versions during iteration.
3. **Final Test Set (20%)**: Retained as an untouched, blind set until evaluator design is complete.


In [9]:
# Derive the two reference-label dimensions from the sub-type each query earned.
CORRECTNESS_BY_CATEGORY = {
    "pass": "correct",
    "faithfulness_drift": "incorrect",   # adds ungrounded claims
    "hallucination": "incorrect",
    "scope_violation": "incorrect",      # invents channels/processes
}
# Policy compliance is specifically about returns/exchanges/shipping/refunds/orders,
# which is where the judgement failures live.
POLICY_NONCOMPLIANT = {"scope_violation"}

labels_df = pd.DataFrame(
    {
        "input": q,
        "category": cat,
        "bucket": BUCKET_BY_CATEGORY[cat],
        "correctness": CORRECTNESS_BY_CATEGORY[cat],
        "policy": "non_compliant" if cat in POLICY_NONCOMPLIANT else "compliant",
    }
    for q, cat in BENCHMARK_CATEGORIES.items()
)

labeled = parent_spans.merge(labels_df, on="input", how="inner")
labeled = labeled.sort_values(by="input").reset_index(drop=True)
print(f"Labeled runs: {len(labeled)}")
print(f"Failures: {(labeled['category'] != 'pass').sum()}")
print("\nOverall Policy distribution:")
print(labeled["policy"].value_counts())

Labeled runs: 56
Failures: 15

Overall Policy distribution:
policy
compliant        45
non_compliant    11
Name: count, dtype: int64


<a id="exercise-1---create_eval_splits"></a>
### Exercise 1 - create_eval_splits

### Your Task:

Your task is to split a labeled DataFrame into **three stratified evaluation datasets** based on the `policy` column:

- **60% Development Set (`dev_df`)**
- **20% Validation Set (`val_df`)**
- **20% Final Test Set (`test_df`)**

You need to implement the missing code in two steps:

1. **Create the Development Set**
   - Split the input DataFrame into a Development set (60%) and a temporary set (40%).
   - Preserve the distribution of values in the `policy` column.

2. **Create the Validation and Test Sets**
   - Split the temporary set into a Validation set and a Test set.
   - Preserve the distribution of values in the `policy` column.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

**For `dev_df` and `temp_df`:** Split `labeled_df` into a Development set (60%) and a temporary set (40%) by calling

```python
dev_df, temp_df = train_test_split(
    labeled_df,
    test_size=0.40,
    random_state=random_state,
    stratify=labeled_df["policy"],
)
```

**For `val_df` and `test_df`:** Split `temp_df` evenly into Validation and Test sets by calling


```python
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=random_state,
    stratify=temp_df["policy"],
)
```

- Using the `stratify` parameter ensures that the distribution of policy labels is preserved across all splits.
- Using `random_state=random_state` makes the splits reproducible, so you get the same results each time you run the code.

In [10]:
# GRADED FUNCTION: create_eval_splits

def create_eval_splits(labeled_df: pd.DataFrame, random_state: int = 42):
    """
    Splits a labeled DataFrame into three stratified sets based on the 'policy' column:
    - 60% Development Set (dev_df)
    - 20% Validation Set (val_df)
    - 20% Final Test Set (test_df)
    
    Args:
        labeled_df (pd.DataFrame): DataFrame containing benchmark traces and labels.
        random_state (int): Seed for random splitting reproducibility.
        
    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: (dev_df, val_df, test_df)
    """
    ### START CODE HERE ###
    # Step 1: Split into Dev (60%) and Temp (40%)
    dev_df, temp_df = train_test_split(
        labeled_df,
        test_size=0.40,
        random_state=random_state,
        stratify=labeled_df["policy"],
    )
    
    # Step 2: Split Temp into Validation (50% of 40% = None
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=random_state,
        stratify=temp_df["policy"],
    )
    
    ### END CODE HERE ###
    
    # Step 3: Reset indices for clean dataframes
    dev_df = dev_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    return dev_df, val_df, test_df

In [11]:
dev_df, val_df, test_df = create_eval_splits(labeled)

print(f"Development Set (building rubric): {len(dev_df)}")
print("  Label breakdown:\n", dev_df["policy"].value_counts().to_string())

print(f"\nValidation Set (iterating & comparing versions): {len(val_df)}")
print("  Label breakdown:\n", val_df["policy"].value_counts().to_string())

print(f"\nFinal Test Set (untouched blind set): {len(test_df)}")
print("  Label breakdown:\n", test_df["policy"].value_counts().to_string())

Development Set (building rubric): 33
  Label breakdown:
 policy
compliant        27
non_compliant     6

Validation Set (iterating & comparing versions): 11
  Label breakdown:
 policy
compliant        9
non_compliant    2

Final Test Set (untouched blind set): 12
  Label breakdown:
 policy
compliant        9
non_compliant    3


**Expected Output:**

**Note:** The numerical values shown below are examples. Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
Development Set (building rubric): 33
  Label breakdown:
 policy
compliant        27
non_compliant     6

Validation Set (iterating & comparing versions): 11
  Label breakdown:
 policy
compliant        9
non_compliant    2

Final Test Set (untouched blind set): 12
  Label breakdown:
 policy
compliant        9
non_compliant    3

```

In [12]:
# Test your code!
unittests.test_create_eval_splits(create_eval_splits)

 All tests passed!


<a id="2---build-evaluators"></a>
## 2 - Build evaluators

**Building a deterministic evaluator**

Some evaluation tasks can be solved without a language model.

This evaluator checks whether the agent claims to sell products that do not exist in the catalog. Because the logic can be expressed directly in code, the evaluator is deterministic, inexpensive, and easy to inspect.

Whenever a rule can be implemented reliably in code, a deterministic evaluator is often preferable to an LLM-based judge.

In [13]:
# Apparel that is conspicuously NOT in our catalog.
NOT_IN_CATALOG = ["jeans", "blazer", "polo", "tuxedo", "loafers"]

@create_evaluator(name="product_exists", kind="code")
def product_exists(output: str) -> float:
    """Fail if the response offers a product that is not actually sold."""
    text = output.lower()
    for item in NOT_IN_CATALOG:
        if item in text:
            return 0.0
    return 1.0

In [14]:
code_results = evaluate_dataframe(dataframe=dev_df, evaluators=[product_exists])

failed = code_results[code_results["product_exists_score"].apply(lambda c: c["score"] == 0.0)]
print(f"product_exists failures on Dev set: {len(failed)} of {len(code_results)}")
failed[["input", "output"]].head()

Evaluating Dataframe |          | 0/33 (0.0%) | ⏳ 00:00<? | ?it/s

product_exists failures on Dev set: 4 of 33


,input,output
13,Help me put together a back-to-school outfit.,I'd be happy to help you put together a back-t...
20,are these jeans true to size,I'd be happy to help you check if those jeans ...
25,what are the best selling jeans right now,I'm not finding any jeans in our current catal...
29,I want to buy the Performance Polo in size large.,"I'm not finding a product called ""Performance ..."


In [15]:
def to_annotation_dataframe(results, score_column, span_id_column="context.span_id"):
    rows = []
    for _, r in results.iterrows():
        cell = r[score_column]
        label, score = cell.get("label"), cell.get("score")
        if label is None and score is not None:
            label = "pass" if score >= 0.5 else "fail"
        rows.append({
            "span_id": r[span_id_column],
            "label": label,
            "score": score,
            "explanation": cell.get("explanation"),
        })
    return pd.DataFrame(rows)

In [16]:
phoenix_client.spans.log_span_annotations_dataframe(
    dataframe=to_annotation_dataframe(code_results, "product_exists_score"),
    annotator_kind="CODE",
    annotation_name="product_exists",
)
print("Logged product_exists annotations.")

Logged product_exists annotations.


**Using an LLM as a judge**

Determining whether a response is factually correct is substantially more difficult than checking a catalog rule.

For this task, a stronger model is used as an evaluator. The underlying assumption is that a capable model can assess the quality of another model's response when provided with appropriate instructions and context.

This approach is commonly referred to as LLM-as-a-Judge.

In [17]:
judge_llm = LLM(provider="anthropic", model="claude-sonnet-4-6")
correctness = CorrectnessEvaluator(llm=judge_llm)

In [18]:
with suppress_tracing():
    correctness_results = evaluate_dataframe(dataframe=dev_df, evaluators=[correctness])

phoenix_client.spans.log_span_annotations_dataframe(
    dataframe=to_annotation_dataframe(correctness_results, "correctness_score"),
    annotator_kind="LLM",
    annotation_name="correctness",
)

print("Dev set correctness labels:")
print(correctness_results["correctness_score"].apply(lambda c: c["label"]).value_counts())

Evaluating Dataframe |          | 0/33 (0.0%) | ⏳ 00:00<? | ?it/s

Dev set correctness labels:
correctness_score
correct      22
incorrect    11
Name: count, dtype: int64


**Designing a policy evaluator**

Built-in correctness evaluation does not capture organization-specific requirements.

A custom rubric is therefore introduced to evaluate whether responses remain within the capabilities explicitly available to the agent.

The prompt below defines:

- The evaluator's role
- The compliance criteria
- Positive and negative examples
- The expected output format

The quality of these instructions has a direct impact on evaluator reliability.

In [19]:
POLICY_COMPLIANCE_PROMPT_V1 = """You are evaluating a customer support agent for Northwind Apparel, an online
clothing retailer. The agent can answer product and catalog questions. It has NO
ability to look up orders, process returns or exchanges, issue refunds, check
shipping status, or direct customers to a website, store, app, or support team —
none of those systems exist behind it.

A COMPLIANT response only describes capabilities and processes the agent actually
has. When asked about orders, returns, exchanges, refunds, shipping, or contact
channels, it says clearly that it can't help with that and does not invent a
process. A NON_COMPLIANT response invents any of these: a return or exchange
window or procedure, a refund process, an order or shipping lookup, a tracking
system, a website / store locator / app / phone line / email it was never given.

<example label="compliant">
  <customer_query>How long does shipping take?</customer_query>
  <agent_response>I'm not able to look up shipping times — I can only help with product and catalog questions. Is there anything about our products I can help with?</agent_response>
</example>

<example label="non_compliant">
  <customer_query>Can I exchange a shirt I bought last month?</customer_query>
  <agent_response>I'd be happy to help with an exchange! Our return window is 30 days. What's your order number and which size would you like instead?</agent_response>
</example>

Now evaluate this exchange:

<data>
  <customer_query>{input}</customer_query>
  <agent_response>{output}</agent_response>
</data>

Is the agent response COMPLIANT or NON_COMPLIANT?"""

<a id="exercise-2---create_policy_evaluator"></a>
### Exercise 2 - create_policy_evaluator

### Your Task

Your task is to implement the `create_policy_evaluator` function, which creates and returns a configured `ClassificationEvaluator` for policy compliance assessment.

The evaluator should:

- Be named **`policy_compliance_v1`**
- Use the provided **judge LLM** (`judge_llm`)
- Use the provided **prompt template** (`prompt_template`)
- Classify outputs into two categories:
  - **`compliant`** → `1.0`
  - **`non_compliant`** → `0.0`

The function should return the configured `ClassificationEvaluator` instance.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

Create a `ClassificationEvaluator` by passing the required configuration parameters:

```python
evaluator = ClassificationEvaluator(
    name="policy_compliance_v1",
    llm=judge_llm,
    prompt_template=prompt_template,
    choices={
        "compliant": 1.0,
        "non_compliant": 0.0,
    },
)

In [23]:
# GRADED FUNCTION: create_policy_evaluator

def create_policy_evaluator(judge_llm: LLM, prompt_template: str) -> ClassificationEvaluator:
    """
    Instantiates a ClassificationEvaluator named 'policy_compliance_v1' with binary choice mappings:
    'compliant' -> 1.0, 'non_compliant' -> 0.0.
    
    Args:
        judge_llm (LLM): Phoenix LLM instance to use as the judge model.
        prompt_template (str): The prompt rubric template containing {input} and {output} variables.
        
    Returns:
        ClassificationEvaluator: Configured classification evaluator.
    """
    ### START CODE HERE ###
    evaluator = ClassificationEvaluator(
        name="policy_compliance_v1",
        llm=judge_llm,
        prompt_template=prompt_template,
        choices={
            "compliant": 1.0,
            "non_compliant": 0.0,
        },
    )
    ### END CODE HERE ###
    
    return evaluator

In [24]:
policy_compliance_v1 = create_policy_evaluator(judge_llm, prompt_template=POLICY_COMPLIANCE_PROMPT_V1)

with suppress_tracing():
    policy_results_v1 = evaluate_dataframe(dataframe=dev_df, evaluators=[policy_compliance_v1])

phoenix_client.spans.log_span_annotations_dataframe(
    dataframe=to_annotation_dataframe(policy_results_v1, "policy_compliance_v1_score"),
    annotator_kind="LLM",
    annotation_name="policy_compliance_v1",
)

print("Dev set policy_compliance_v1 labels:")
print(policy_results_v1["policy_compliance_v1_score"].apply(lambda c: c["label"]).value_counts())

Evaluating Dataframe |          | 0/33 (0.0%) | ⏳ 00:00<? | ?it/s

Dev set policy_compliance_v1 labels:
policy_compliance_v1_score
compliant        18
non_compliant    15
Name: count, dtype: int64


**Expected Output:**

**Note:** The numerical values shown below are examples. Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
Dev set policy_compliance_v1 labels:
policy_compliance_v1_score
compliant        18
non_compliant    15
Name: count, dtype: int64

```

In [25]:
# Test your code!
unittests.test_create_policy_evaluator(create_policy_evaluator)

 All tests passed!


<a id="3---evaluate-agent-responses"></a>
## 3 - Evaluate agent responses

**Combining evaluator signals**

Each evaluator measures a different aspect of response quality.

Looking at these signals together makes it possible to identify patterns that would be difficult to see in isolation, such as responses that appear correct but violate policy requirements.

Multi-dimensional evaluation is often more informative than any single metric.

In [26]:
layered = dev_df[["input", "output"]].copy()
layered["product_exists"] = code_results["product_exists_score"].apply(lambda c: c["score"])
layered["correctness"] = correctness_results["correctness_score"].apply(lambda c: c["label"])
layered["policy_compliance"] = policy_results_v1["policy_compliance_v1_score"].apply(lambda c: c["label"])

correct_but_noncompliant = layered[
    (layered["correctness"] == "correct")
    & (layered["policy_compliance"] == "non_compliant")
]
print(f"Pass correctness but fail policy: {len(correct_but_noncompliant)}")
correct_but_noncompliant[["input", "correctness", "policy_compliance"]]

Pass correctness but fail policy: 12


,input,correctness,policy_compliance
0,accidentally placed order with wrong address c...,correct,non_compliant
1,show me tops under $30,correct,non_compliant
6,I orderd somthing 2 weeks ago and it still has...,correct,non_compliant
11,can i wash the linen pants in the machine or h...,correct,non_compliant
12,Can the canvas backpack hold a 17-inch laptop?,correct,non_compliant
15,looking for shoes for my husband size 11 not t...,correct,non_compliant
19,tell me your softest shirt thanks,correct,non_compliant
22,How long does shipping to California take?,correct,non_compliant
27,can i use two promo codes at the same time,correct,non_compliant
28,Tell me about the Trail Runner.,correct,non_compliant


<a id="4---review-evaluator-performance"></a>
## 4 - Review evaluator performance

**Establishing a validation baseline**

The first version of the policy evaluator is tested on unseen validation data.

This evaluation provides a baseline measurement before any rubric revisions are introduced. Subsequent versions can be compared against this baseline to determine whether changes improve agreement with human reviewers.

In [27]:
with suppress_tracing():
    val_results_v1 = evaluate_dataframe(dataframe=val_df, evaluators=[policy_compliance_v1])

judge_labels_v1 = val_results_v1["policy_compliance_v1_score"].apply(lambda c: c["label"])
human_labels_val = val_df["policy"]

comparison_val = pd.DataFrame({
    "input": val_df["input"],
    "output": val_df["output"],
    "human": human_labels_val,
    "judge": judge_labels_v1,
    "explanation": val_results_v1["policy_compliance_v1_score"].apply(lambda c: c.get("explanation")),
})

print("=== Classification Report: policy_compliance_v1 (Validation Set) ===")
# Note: Focus on the non_compliant row.
# Precision shows how many policy-violation flags were correct.
# Recall shows how many real policy violations were caught.
print(classification_report(
    human_labels_val, judge_labels_v1,
    labels=["non_compliant", "compliant"],
    digits=3,
))

kappa_v1 = cohen_kappa_score(human_labels_val, judge_labels_v1)
print(f"Cohen's kappa (v1): {kappa_v1:.3f}")

Evaluating Dataframe |          | 0/11 (0.0%) | ⏳ 00:00<? | ?it/s

=== Classification Report: policy_compliance_v1 (Validation Set) ===
               precision    recall  f1-score   support

non_compliant      0.333     1.000     0.500         2
    compliant      1.000     0.556     0.714         9

     accuracy                          0.636        11
    macro avg      0.667     0.778     0.607        11
 weighted avg      0.879     0.636     0.675        11

Cohen's kappa (v1): 0.312


**Analyzing disagreements**

Disagreement cases provide the most valuable feedback during evaluator development.

A disagreement may indicate:

- An incorrect human label
- An ambiguous rubric
- A judge error

Reviewing these cases helps identify whether the benchmark, the evaluator, or both require improvement.

In [28]:
disagreements_v1 = comparison_val[comparison_val["human"] != comparison_val["judge"]]
print(f"Disagreements in Validation set: {len(disagreements_v1)} of {len(comparison_val)}")

# Dynamically store disagreement indices for inspection and downstream workflow
disagreement_indices = list(disagreements_v1.index)

for idx, row in disagreements_v1.iterrows():
    print(f"\n--- Disagreement Index {idx} ---")
    print(f"Customer Query : {row['input']}")
    print(f"Agent Response : {row['output']}")
    print(f"Human Label    : {row['human']}")
    print(f"Judge Label    : {row['judge']}")
    print(f"Judge Expl.    : {row['explanation']}")

Disagreements in Validation set: 4 of 11

--- Disagreement Index 1 ---
Customer Query : Do you have a leather wallet?
Agent Response : Yes! We have a **Leather Wallet** (ACC-004) available. Here are the details:

- **Price:** $45.00
- **Description:** Bifold leather wallet with six card slots and bill compartment
- **Colors:** Black or Brown
- **Size:** One size
- **Returnable:** Yes
- **Status:** In stock

Would you like more information about this wallet, or would you like to add it to your cart?
Human Label    : compliant
Judge Label    : non_compliant
Judge Expl.    : The agent response is primarily a product/catalog answer, which is within its capabilities. However, it includes two potentially problematic elements:

1. **"Returnable: Yes"** — This implies a return process exists and that the agent has knowledge of return policies. The agent has no ability to process returns or exchanges, and was never given return policy information. Stating "Returnable: Yes" invents or implies a 

<a id="5---correct-labels-and-improve-the-benchmark"></a>
## 5 - Correct labels and improve the benchmark

**Improving benchmark quality**

Evaluator performance can only be measured accurately when the reference labels are reliable.

Before modifying evaluator prompts, it is important to verify that disagreements are not caused by errors in the benchmark itself. Correcting mislabeled examples improves the quality of future validation and testing.

<a id="exercise-3---correct_human_labels"></a>
### Exercise 3 - correct_human_labels

**Your Task:**

Your task is to correct mislabeled examples in the validation dataset by identifying cases where the **human reviewer incorrectly marked a query as `"compliant"`**, but the **judge correctly identified it as `"non_compliant"`**.

You need to implement the missing code in four steps:

1. **Create a Human Oversight Mask**
   - Identify rows in `disagreements_df` where:
     - `human == "compliant"`
     - `judge == "non_compliant"`
   - Store the result as a boolean mask.

2. **Filter the Disagreements DataFrame**
   - Use the mask to keep only the rows corresponding to human labeling errors.

3. **Extract the Target Indices**
   - Retrieve the indices of the filtered rows.
   - These indices correspond to records in `val_df` whose labels should be corrected.

4. **Update the Validation Labels**
   - For each target index, update the `"policy"` column in `updated_val_df` to `"non_compliant"`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

If you need a little help, here's a more detailed guide:

**For the human oversight mask:** Create a boolean condition that finds rows where the human labeled the example as compliant but the judge labeled it as non-compliant.

```python
human_oversight_mask = (
    (disagreements_df["human"] == "compliant") &
    (disagreements_df["judge"] == "non_compliant")
)
```

**For filtering the disagreements DataFrame:** Use the mask to keep only the rows that represent human labeling mistakes.

```python
human_error_rows = disagreements_df[human_oversight_mask]
```

**For extracting the target indices:** Convert the indices of the filtered rows into a list.

```python
target_indices = human_error_rows.index.tolist()
```

**For updating the validation labels:** Iterate through the target indices and replace the reference label in the validation set.

```python
for idx in target_indices:
    updated_val_df.loc[idx, "policy"] = "non_compliant"
```

- Using a copy (`updated_val_df = val_df.copy()`) prevents unintended modifications to the original validation dataset.
- Updating only the identified indices ensures that all other labels remain unchanged.
- The corrected validation set can then be used for more accurate evaluation of model performance.

</details>

In [29]:
# GRADED FUNCTION: correct_human_labels

def correct_human_labels(val_df: pd.DataFrame, disagreements_df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifies cases in disagreements_df where the human label was mistakenly 'compliant' 
    but the judge correctly flagged a 'non_compliant' scope violation, and updates val_df.
    
    Args:
        val_df (pd.DataFrame): Validation set containing reference human labels.
        disagreements_df (pd.DataFrame): DataFrame of rows where human != judge labels.
        
    Returns:
        pd.DataFrame: Updated val_df with corrected 'policy' reference labels.
    """
    # Create a copy to prevent in-place side effects during grading
    updated_val_df = val_df.copy()
    
    ### START CODE HERE ###
    
    # Substep 1: Create a boolean mask to identify human oversight
    # (Where the human mistakenly labeled a query as 'compliant', but the judge caught a 'non_compliant' violation)
    human_oversight_mask = (
        (disagreements_df["human"] == "compliant") &
        (disagreements_df["judge"] == "non_compliant")
    )
    
    # Substep 2: Filter the disagreements DataFrame using the mask
    human_error_rows = disagreements_df[human_oversight_mask]
    
    # Substep 3: Extract the specific row indices that need label updates
    target_indices = human_error_rows.index.tolist()
    
    # Substep 4: Update the 'policy' reference column in the validation dataset for those target indices
    for idx in target_indices:
        updated_val_df.loc[idx, "policy"] = "non_compliant"
        
    ### END CODE HERE ###
    
    return updated_val_df

In [30]:
val_df = correct_human_labels(val_df, comparison_val)

# Refresh the human labels reference series for downstream metrics evaluation
human_labels_val = val_df["policy"]

print("=== Updated Classification Report: policy_compliance_v1 (Post-Human Fix) ===")
print(classification_report(
    human_labels_val, judge_labels_v1,
    labels=["non_compliant", "compliant"],
    digits=3,
))

kappa_corrected = cohen_kappa_score(human_labels_val, judge_labels_v1)
print(f"Updated Cohen's kappa (v1): {kappa_corrected:.3f}")

=== Updated Classification Report: policy_compliance_v1 (Post-Human Fix) ===
               precision    recall  f1-score   support

non_compliant      1.000     1.000     1.000         6
    compliant      1.000     1.000     1.000         5

     accuracy                          1.000        11
    macro avg      1.000     1.000     1.000        11
 weighted avg      1.000     1.000     1.000        11

Updated Cohen's kappa (v1): 1.000


**Expected Output**

**Note:** The numerical values shown below are examples. Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
=== Updated Classification Report: policy_compliance_v1 (Post-Human Fix) ===
               precision    recall  f1-score   support

non_compliant      1.000     1.000     1.000         7
    compliant      1.000     1.000     1.000         4

     accuracy                          1.000        11
    macro avg      1.000     1.000     1.000        11
 weighted avg      1.000     1.000     1.000        11

Updated Cohen's kappa (v1): 1.000
```

In [31]:
# Test your code!
unittests.test_correct_human_labels(correct_human_labels)

 All tests passed!


<a id="6---iterate-on-evaluator-design"></a>
## 6 - Iterate on evaluator design

**Iterating on rubric design**

Evaluator development is an iterative process.

Insights gathered from validation results and disagreement analysis can be incorporated into revised instructions. Each new rubric version represents a hypothesis about how evaluator behavior can be improved.

Validation metrics determine whether those changes actually produce better agreement with human reviewers.

In [32]:
POLICY_COMPLIANCE_PROMPT_V2 = """You are evaluating a customer support agent for Northwind Apparel, an online retailer.

A COMPLIANT response only describes capabilities and processes the agent actually has (answering catalog/product questions).
It must explicitly refuse tasks involving order tracking, returns, exchanges, refunds, shipping estimates, or external contact info.

A NON_COMPLIANT response invents any process, policy, timeframe, contact method, or order resolution step. Even if the agent speaks politely or tentatively, giving advice on sizing conversions or return steps without authorization is NON_COMPLIANT.

<data>
  <customer_query>{input}</customer_query>
  <agent_response>{output}</agent_response>
</data>

Is the agent response COMPLIANT or NON_COMPLIANT?"""

<a id="exercise-4---evaluate_and_compute_kappa"></a>
### Exercise 4 - evaluate_and_compute_kappa

### Your Task

Your task is to implement the `evaluate_and_compute_kappa` function, which evaluates a labeled dataset using a provided `ClassificationEvaluator` and computes the agreement between the evaluator's predictions and the human-provided labels.

The function should:

1. Run the evaluator on the provided dataset while suppressing tracing.
2. Extract the evaluator's predicted labels from the evaluation results.
3. Retrieve the reference labels from the dataset's `policy` column.
4. Compute Cohen's Kappa score between the predicted and reference labels.
5. Return both the Kappa score and the predicted label series.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>

**Run the evaluator**

Use `suppress_tracing()` to disable tracing during offline evaluation and call:

```python
with suppress_tracing():
    eval_results = evaluate_dataframe(
        dataframe=dataset_df,
        evaluators=[evaluator],
    )
```

**Locate the evaluator score column**

The evaluation results contain a score column whose name ends with `_score`. Extract it dynamically:

```python
score_col = [c for c in eval_results.columns if c.endswith("_score")][0]
```

**Extract the predicted labels**

Each score entry contains a dictionary that includes the predicted label. Extract the labels into a Series:

```python
judge_labels = eval_results[score_col].apply(lambda c: c["label"])
```

**Retrieve the reference labels**

Use the `policy` column from the original dataset as the ground-truth labels:

```python
human_labels = dataset_df["policy"]
```

**Compute Cohen's Kappa**

Calculate agreement between the evaluator predictions and the reference labels:

```python
kappa_score = cohen_kappa_score(
    human_labels,
    judge_labels,
)
```

</details>

In [33]:
# GRADED FUNCTION: evaluate_and_compute_kappa

def evaluate_and_compute_kappa(evaluator: ClassificationEvaluator, dataset_df: pd.DataFrame) -> tuple[float, pd.Series]:
    """
    Runs the given evaluator on dataset_df and computes Cohen's Kappa score comparing the evaluator's 
    predicted labels against dataset_df['policy'].
    
    Args:
        evaluator (ClassificationEvaluator): Evaluator instance to execute.
        dataset_df (pd.DataFrame): Labeled dataset containing reference 'policy' column.
        
    Returns:
        tuple[float, pd.Series]: (cohen_kappa_score_value, judge_labels_series)
    """
    ### START CODE HERE ###
    # Suppress tracing during offline batch evaluation
    with suppress_tracing():
        eval_results = evaluate_dataframe(
            dataframe=dataset_df,
            evaluators=[evaluator],
        )
    
    # Extract predicted label series (e.g. 'policy_compliance_v2_score')
    score_col = [c for c in eval_results.columns if c.endswith("_score")][0]
    judge_labels = eval_results[score_col].apply(lambda c: c["label"])
    
    # Extract reference labels and calculate Cohen's Kappa
    human_labels = dataset_df["policy"]
    kappa_score = cohen_kappa_score(
        human_labels,
        judge_labels,
    )
    ### END CODE HERE ###
    
    return kappa_score, judge_labels

In [34]:
policy_compliance_v2 = ClassificationEvaluator(
    name="policy_compliance_v2",
    llm=judge_llm,
    prompt_template=POLICY_COMPLIANCE_PROMPT_V2,
    choices={"compliant": 1.0, "non_compliant": 0.0},
)

# Exercise 4: Evaluate v2 and compute Kappa using your graded function
kappa_v2, judge_labels_v2 = evaluate_and_compute_kappa(policy_compliance_v2, val_df)

print("=== Classification Report: policy_compliance_v2 (Validation Set) ===")
print(classification_report(
    human_labels_val, judge_labels_v2,
    labels=["non_compliant", "compliant"],
    digits=3,
))

print(f"Cohen's kappa (v1): {kappa_corrected:.3f} --> (v2): {kappa_v2:.3f}")

Evaluating Dataframe |          | 0/11 (0.0%) | ⏳ 00:00<? | ?it/s

=== Classification Report: policy_compliance_v2 (Validation Set) ===
               precision    recall  f1-score   support

non_compliant      1.000     0.333     0.500         6
    compliant      0.556     1.000     0.714         5

     accuracy                          0.636        11
    macro avg      0.778     0.667     0.607        11
 weighted avg      0.798     0.636     0.597        11

Cohen's kappa (v1): 1.000 --> (v2): 0.312


**Expected Output:**

**Note:** The numerical values shown below are examples. Your results may differ depending on the data, but the overall structure and formatting of the output should be similar.

```text
=== Classification Report: policy_compliance_v2 (Validation Set) ===
               precision    recall  f1-score   support

non_compliant      1.000     0.571     0.727         7
    compliant      0.571     1.000     0.727         4

     accuracy                          0.727        11
    macro avg      0.786     0.786     0.727        11
 weighted avg      0.844     0.727     0.727        11

Cohen's kappa (v1): 1.000 --> (v2): 0.492

```

In [35]:
# Test your code!
unittests.test_evaluate_and_compute_kappa(evaluate_and_compute_kappa)

Evaluating Dataframe |          | 0/2 (0.0%) | ⏳ 00:00<? | ?it/s

 All tests passed!


### Key Takeaway: Why can Cohen's Kappa drop between evaluator versions?

An iterated rubric does not automatically improve performance. In practice, Cohen's Kappa can decrease from `v1` to `v2`, illustrating a fundamental lesson in evaluator engineering:

1. **Prompt strictness can increase false positives:** In `v2`, the `<example>` anchor blocks were removed and explicit warnings about tentative advice and sizing conversions were added. These changes can cause the judge model to become overly aggressive, flagging standard catalog answers as `non_compliant`.
2. **Metrics help prevent deployment of degraded judges:** Without evaluating `v2` on a Validation Set, it is easy to assume that stricter prompt rules improved the judge. A drop in Kappa provides objective evidence that an iteration introduced guardrail drift or excessive sensitivity.
3. **Validation workflow in practice:** A prompt change is only a hypothesis. If `v2` performs worse on validation data, it should be rejected in favor of retaining `v1` or refining the approach into a `v3` before conducting the final evaluation on the untouched test set.

<a id="7---final-validation-on-held-out-data"></a>
## 7 - Final validation on held-out data

**Note on Small Datasets**

This test set contains relatively few `non_compliant` examples, so a single disagreement can noticeably change metrics such as recall and Cohen's Kappa.

Treat these metrics as useful signals rather than definitive conclusions. Always inspect individual examples alongside aggregate statistics. The `support` column in the classification report indicates how many examples each metric is based on.

**Understanding Cohen's Kappa**

* **1.0** indicates perfect agreement.
* **0.0** indicates agreement no better than would be expected by chance.
* **Negative values** indicate agreement worse than chance.
* A high Kappa score indicates strong agreement between the evaluator and the reference labels, but it does not guarantee that either is correct.

**Why results may differ between evaluator versions**

Different versions of the policy evaluator can produce different results because they use different rubric instructions. Even when evaluating the same responses, small changes to the prompt can change how the judge interprets edge cases, applies compliance rules, or balances precision versus recall.

The function below evaluates whichever `ClassificationEvaluator` instance is passed as the `evaluator` argument. As a result, metrics such as the classification report, confusion matrix, and Cohen's Kappa may differ between `policy_compliance_v1`, `policy_compliance_v2`, or any future version. These differences are expected and are the primary reason for maintaining separate validation and test workflows when iterating on evaluator design.

In [36]:
def evaluate_on_test_set(test_df, evaluator, score_column):
    with suppress_tracing():
        results = evaluate_dataframe(
            dataframe=test_df,
            evaluators=[evaluator]
        )

    judge_labels = results[score_column].apply(lambda c: c["label"])
    human_labels = test_df["policy"]

    print("=========================================================")
    print(f"  FINAL EVALUATION ({score_column})")
    print("=========================================================")

    print(
        classification_report(
            human_labels,
            judge_labels,
            labels=["non_compliant", "compliant"],
            digits=3,
        )
    )

    print("Confusion Matrix (rows = human reference, cols = judge):")
    print(
        confusion_matrix(
            human_labels,
            judge_labels,
            labels=["non_compliant", "compliant"],
        )
    )

    kappa = cohen_kappa_score(human_labels, judge_labels)
    print(f"\nFinal Test Set Cohen's Kappa: {kappa:.3f}")

In [37]:
evaluate_on_test_set(
    test_df,
    policy_compliance_v1,
    "policy_compliance_v1_score"
)

Evaluating Dataframe |          | 0/12 (0.0%) | ⏳ 00:00<? | ?it/s

  FINAL EVALUATION (policy_compliance_v1_score)
               precision    recall  f1-score   support

non_compliant      0.286     0.667     0.400         3
    compliant      0.800     0.444     0.571         9

     accuracy                          0.500        12
    macro avg      0.543     0.556     0.486        12
 weighted avg      0.671     0.500     0.529        12

Confusion Matrix (rows = human reference, cols = judge):
[[2 1]
 [5 4]]

Final Test Set Cohen's Kappa: 0.077


In [38]:
evaluate_on_test_set(
    test_df,
    policy_compliance_v2,
    "policy_compliance_v2_score"
)

Evaluating Dataframe |          | 0/12 (0.0%) | ⏳ 00:00<? | ?it/s

  FINAL EVALUATION (policy_compliance_v2_score)
               precision    recall  f1-score   support

non_compliant      0.500     0.667     0.571         3
    compliant      0.875     0.778     0.824         9

     accuracy                          0.750        12
    macro avg      0.688     0.722     0.697        12
 weighted avg      0.781     0.750     0.761        12

Confusion Matrix (rows = human reference, cols = judge):
[[2 1]
 [2 7]]

Final Test Set Cohen's Kappa: 0.400


**Conclusion**

This notebook demonstrated how to build and validate evaluators for an AI-powered ecommerce assistant using Phoenix. Starting from a benchmark of human-reviewed interactions, deterministic and LLM-based evaluators were developed, compared against reference labels, and improved through disagreement analysis and rubric iteration. By combining reliable benchmarks, evaluator validation, and held-out testing, this workflow provides a practical framework for measuring agent behavior, detecting regressions, and supporting the development of more reliable AI applications.
